Poker ML Pipeline - Data Collection

Collects poker hand history data from GitHub repository and saves to Workspace.

Prerequisites: Serverless compute attached

In [ ]:
# Install tomli for TOML parsing (Python <3.11)
%pip install tomli -q

In [ ]:
# Import all required libraries for data processing, API calls, and file handling
from dataclasses import dataclass, asdict
from typing import List, Optional, Dict, Any, Iterable
from functools import lru_cache
from uuid import uuid4
from urllib.parse import urlparse
from pathlib import Path
import os
import re
import time
import pandas as pd
import requests

try:
    import tomllib
except ImportError:
    import tomli as tomllib

print("=" * 80)
print("SP-1: Poker Hand History Data Collection")
print("=" * 80)
start_time = time.time()

In [ ]:
# Configuration: Set GitHub repository details and output paths
GITHUB_REPO = "ineedcoldbrew/UofToronto-poker-dataset"
GITHUB_BASE_PATH = "data/handhq"
NUM_PARENT_FOLDERS = 50
NUM_FILES_PER_FOLDER = 200
HAND_LIMIT_PER_FILE = None
OUT_PREFIX = "phh_multi"

# Use Workspace files for persistent storage in Databricks Serverless
OUTPUT_DIR = Path('/Workspace/Users/leo.lwakabamba@gmail.com/poker-ml-data/')
GITHUB_TOKEN = None

print(f"Output directory: {OUTPUT_DIR}")
print(f"Will collect {NUM_FILES_PER_FOLDER} files from {NUM_PARENT_FOLDERS} folder(s)")

In [ ]:
# Data Models: Define dataclasses for poker hand data structures
@dataclass
class HandRow:
    hand_id: str
    variant: Optional[str]
    min_bet: Optional[float]
    currency: Optional[str]
    event: Optional[str]
    year: Optional[int]
    venue: Optional[str]
    sb: Optional[float]
    bb: Optional[float]
    straddle: Optional[float]
    table_name: Optional[str]
    time: Optional[str]
    day: Optional[int]
    month: Optional[int]
    hand_no: Optional[int]
    tz: Optional[str]
    button_seat: Optional[int]  # NEW: Button seat number for position calculation
    num_players: Optional[int]  # NEW: Number of players at the table

@dataclass
class SeatRow:
    hand_id: str
    seat_no: int
    player_name: Optional[str]
    starting_stack: Optional[float]
    ante: Optional[float]
    position_from_button: Optional[int]  # NEW: 0=BTN, 1=SB, 2=BB, 3=UTG, etc.
    position_name: Optional[str]  # NEW: BTN, SB, BB, UTG, MP, CO, etc.

@dataclass
class BoardRow:
    hand_id: str
    flop: Optional[str]
    turn: Optional[str]
    river: Optional[str]

@dataclass
class ActionRow:
    hand_id: str
    idx: int
    street: str
    actor: Optional[str]
    action_type: str
    amount: Optional[float]
    cards: Optional[str]
    raw: str

@dataclass
class ShowdownRow:
    hand_id: str
    player_name: Optional[str]
    cards: Optional[str]

TABLE_COLUMNS = {
    'hands': list(HandRow.__annotations__.keys()),
    'seats': list(SeatRow.__annotations__.keys()),
    'boards': list(BoardRow.__annotations__.keys()),
    'actions': list(ActionRow.__annotations__.keys()),
    'showdown': list(ShowdownRow.__annotations__.keys()),
}

TABLE_ORDER = ['hands', 'seats', 'boards', 'actions', 'showdown']
print("Data models defined (with position features)")

In [ ]:
# Helper Functions: Functions to fetch and parse PHHS files from GitHub
def to_raw_github(url: str) -> str:
    """Convert GitHub web URL to raw content URL"""
    parsed = urlparse(url)
    if "github.com" not in parsed.netloc:
        return url
    parts = parsed.path.strip('/').split('/')
    if len(parts) < 5:
        raise ValueError(f"Unexpected GitHub URL format: {url}")
    user, repo, _, branch = parts[:4]
    rest = '/'.join(parts[4:])
    return f"https://raw.githubusercontent.com/{user}/{repo}/{branch}/{rest}"

def fetch_text(url: str) -> str:
    """Fetch text content from URL"""
    raw_url = to_raw_github(url)
    response = requests.get(raw_url, timeout=60)
    response.raise_for_status()
    response.encoding = 'utf-8'
    return response.text

TIME_LINE_RE = re.compile(r"(?m)^(time\s*=\s*)(\d{2}:\d{2}:\d{2})\s*$")

def pre_toml_fix(raw_text: str) -> str:
    """Fix time field formatting for TOML parsing"""
    return TIME_LINE_RE.sub(lambda m: f"{m.group(1)}'{m.group(2)}'", raw_text)

def parse_phhs_to_dict(text: str) -> Dict[str, Any]:
    """Parse PHHS text to dictionary"""
    fixed = pre_toml_fix(text)
    return tomllib.loads(fixed)

def parse_actions_list(raw_actions: List[str], seat_map: Dict[str, str]) -> tuple:
    """Parse action strings into structured rows"""
    action_rows: List[ActionRow] = []
    showdown_rows: List[ShowdownRow] = []
    street = 'preflop'
    flop = turn = river = None

    def parse_amt(token: str) -> Optional[float]:
        try:
            return float(token)
        except Exception:
            return None

    for idx, raw in enumerate(raw_actions, start=1):
        tokens = raw.split()
        if len(tokens) >= 3 and tokens[0] == 'd' and tokens[1] == 'dh':
            target = tokens[2]
            cards = tokens[3] if len(tokens) > 3 else None
            action_rows.append(ActionRow(
                hand_id='', idx=idx, street=street, actor='Dealer',
                action_type='deal_hole', amount=None,
                cards=f"{seat_map.get(target, target)}:{cards}" if cards else None,
                raw=raw
            ))
        elif len(tokens) >= 3 and tokens[0] == 'd' and tokens[1] == 'db':
            cards = tokens[2]
            if len(cards) == 6 and flop is None:
                flop = cards
                street = 'flop'
                which = 'deal_flop'
            elif len(cards) == 2 and turn is None:
                turn = cards
                street = 'turn'
                which = 'deal_turn'
            elif len(cards) == 2:
                river = cards
                street = 'river'
                which = 'deal_river'
            else:
                which = 'deal_board'
            action_rows.append(ActionRow('', idx, street, 'Dealer', which, None, cards, raw))
        elif tokens and tokens[0].startswith('p'):
            actor = seat_map.get(tokens[0], tokens[0])
            if len(tokens) == 2 and tokens[1] == 'f':
                action_rows.append(ActionRow('', idx, street, actor, 'fold', None, None, raw))
            elif len(tokens) == 2 and tokens[1] == 'cc':
                action_rows.append(ActionRow('', idx, street, actor, 'call_or_check', None, None, raw))
            elif len(tokens) == 3 and tokens[1] == 'cbr':
                action_rows.append(ActionRow('', idx, street, actor, 'bet_or_raise_to', parse_amt(tokens[2]), None, raw))
            elif len(tokens) == 3 and tokens[1] == 'sm':
                cards = tokens[2]
                showdown_rows.append(ShowdownRow('', actor, cards))
                action_rows.append(ActionRow('', idx, street, actor, 'show', None, cards, raw))
            else:
                action_rows.append(ActionRow('', idx, street, actor, 'unknown', None, None, raw))
        else:
            action_rows.append(ActionRow('', idx, street, None, 'unparsed', None, None, raw))

    board_row = BoardRow('', flop, turn, river)
    return action_rows, showdown_rows, board_row

print("Helper functions loaded")

In [ ]:
# GitHub Repository Discovery: Functions to discover and list PHHS files
GITHUB_API_ROOT = "https://api.github.com"

def github_headers():
    """Generate GitHub API headers with optional token"""
    token = GITHUB_TOKEN or os.environ.get('GITHUB_TOKEN')
    headers = {'Accept': 'application/vnd.github.v3+json'}
    if token:
        headers['Authorization'] = f'Bearer {token}'
    return headers

@lru_cache(maxsize=None)
def list_repo_dir(path: str):
    """List contents of a GitHub repository directory"""
    url = f"{GITHUB_API_ROOT}/repos/{GITHUB_REPO}/contents/{path}"
    response = requests.get(url, headers=github_headers(), timeout=60)
    if response.status_code == 404:
        raise FileNotFoundError(f'Path not found in repo: {path}')
    response.raise_for_status()
    return response.json()

def pick_folders(limit: Optional[int]):
    """Pick top-level folders from the repository"""
    entries = list_repo_dir(GITHUB_BASE_PATH)
    dirs = sorted([e for e in entries if e['type'] == 'dir'], key=lambda e: e['name'])
    return dirs if limit is None else dirs[:limit]

def gather_phhs_files(folder_entry: Dict[str, Any], files_per_folder: Optional[int]):
    """Gather PHHS files from a folder (recursive)"""
    queue = [folder_entry['path']]
    collected = []
    while queue and (files_per_folder is None or len(collected) < files_per_folder):
        current = queue.pop(0)
        entries = list_repo_dir(current)
        entries = sorted(entries, key=lambda e: e['name'])
        for item in entries:
            if item['type'] == 'file' and item['name'].lower().endswith('.phhs'):
                collected.append({'path': item['path'], 'download_url': item['download_url']})
                if files_per_folder is not None and len(collected) >= files_per_folder:
                    break
            elif item['type'] == 'dir':
                queue.append(item['path'])
    return collected

def discover_file_urls(num_folders: Optional[int], files_per_folder: Optional[int]):
    """Discover PHHS file URLs from the repository"""
    folders = pick_folders(num_folders)
    discovered = []
    for folder in folders:
        files = gather_phhs_files(folder, files_per_folder)
        if not files:
            print(f"   Warning: no PHHS files found under {folder['name']}")
        else:
            print(f"   Folder {folder['name']}: taking {len(files)} file(s).")
        discovered.extend(files)
    return discovered

print("Repository discovery functions loaded")

In [ ]:
# Hand Normalization: Convert raw PHHS hand data into structured pandas DataFrames

def get_position_name(position_from_button: int, num_players: int) -> str:
    """
    Convert position_from_button to a human-readable position name.
    
    Position from button:
    - 0 = Button (BTN) - acts last post-flop, best position
    - 1 = Small Blind (SB) - posts small blind
    - 2 = Big Blind (BB) - posts big blind  
    - 3+ = Early to middle positions
    
    The last position before SB depends on table size.
    """
    if position_from_button == 0:
        return 'BTN'
    elif position_from_button == 1:
        return 'SB'
    elif position_from_button == 2:
        return 'BB'
    elif num_players <= 3:
        # Heads-up or 3-handed: only BTN, SB, BB
        return 'BB'  # Shouldn't reach here for 2-3 players
    elif num_players == 4:
        # 4-handed: BTN, SB, BB, UTG
        return 'UTG'
    elif num_players == 5:
        # 5-handed: BTN, SB, BB, UTG, CO
        if position_from_button == 3:
            return 'UTG'
        else:
            return 'CO'
    elif num_players == 6:
        # 6-handed: BTN, SB, BB, UTG, MP, CO
        if position_from_button == 3:
            return 'UTG'
        elif position_from_button == 4:
            return 'MP'
        else:
            return 'CO'
    else:
        # Full ring (7-10 players): BTN, SB, BB, UTG, UTG+1, MP, MP+1, HJ, CO
        positions_before_btn = num_players - 1  # Excluding BTN
        if position_from_button == 3:
            return 'UTG'
        elif position_from_button == 4:
            return 'UTG+1'
        elif position_from_button == positions_before_btn - 1:
            return 'CO'  # Cutoff is always 1 before BTN
        elif position_from_button == positions_before_btn - 2:
            return 'HJ'  # Hijack is 2 before BTN
        else:
            return 'MP'  # Everything else is middle position

def normalize_one_hand(hand_blob: Dict[str, Any]) -> Dict[str, pd.DataFrame]:
    """Turn parsed PHHS hand into flat tables"""
    variant = hand_blob.get('variant')
    min_bet = hand_blob.get('min_bet')
    currency_symbol = hand_blob.get('currency_symbol') or hand_blob.get('currency')
    event = hand_blob.get('event')
    year = hand_blob.get('year')
    venue = hand_blob.get('venue')
    table_name = hand_blob.get('table')
    tz = hand_blob.get('time_zone_abbreviation')
    hh_time = hand_blob.get('time')
    day = hand_blob.get('day')
    month = hand_blob.get('month')
    hand_no = hand_blob.get('hand')

    blinds = hand_blob.get('blinds_or_straddles') or []
    sb = float(blinds[0]) if len(blinds) > 0 else None
    bb = float(blinds[1]) if len(blinds) > 1 else None
    straddle = float(blinds[2]) if len(blinds) > 2 else None

    players = hand_blob.get('players') or []
    stacks = hand_blob.get('starting_stacks') or []
    antes = hand_blob.get('antes') or []
    seats_order = hand_blob.get('seats') or list(range(1, len(players) + 1))
    actions = hand_blob.get('actions') or []
    
    num_players = len(players)
    
    # PHH Format: Last player in the list is the button
    # Position from button: 0=BTN, 1=SB, 2=BB, 3=UTG, etc.
    # The first player (index 0) is SB, last player (index -1) is BTN
    
    # Calculate button seat number (the seat of the last player in the list)
    button_seat = int(seats_order[-1]) if seats_order and len(seats_order) > 0 else None

    hand_id = f"handhq_{uuid4().hex[:8]}"

    hand_row = HandRow(
        hand_id=hand_id, variant=variant,
        min_bet=float(min_bet) if isinstance(min_bet, (int, float)) else None,
        currency=currency_symbol, event=event,
        year=int(year) if isinstance(year, int) else None,
        venue=venue, sb=sb, bb=bb, straddle=straddle,
        table_name=table_name,
        time=str(hh_time) if hh_time is not None else None,
        day=int(day) if isinstance(day, int) else None,
        month=int(month) if isinstance(month, int) else None,
        hand_no=int(hand_no) if isinstance(hand_no, int) else None,
        tz=tz,
        button_seat=button_seat,
        num_players=num_players,
    )

    seat_rows: List[SeatRow] = []
    seat_map: Dict[str, str] = {}
    
    for i, player in enumerate(players):
        seat_no = int(seats_order[i]) if i < len(seats_order) else (i + 1)
        name = str(player) if player is not None else None
        stack = float(stacks[i]) if i < len(stacks) and stacks[i] is not None else None
        ante = float(antes[i]) if i < len(antes) and antes[i] is not None else None
        
        # Calculate position from button
        # PHH format: players[0] = SB (position 1 from BTN)
        #            players[-1] = BTN (position 0 from BTN)
        # So: position_from_button = (num_players - 1 - i) for each player i
        # Wait, that's wrong. Let me recalculate:
        # players[0] = SB = position 1
        # players[1] = BB = position 2
        # players[-1] = BTN = position 0
        # So if we number from BTN: BTN=0, then going backwards:
        # position = (num_players - 1 - i) % num_players... no
        # Actually simpler: 
        # - Player at index (num_players - 1) is BTN (position 0)
        # - Player at index 0 is SB (position 1)
        # - Player at index 1 is BB (position 2)
        # So position_from_button = 1 + i for i < num_players - 1
        # And position_from_button = 0 for i = num_players - 1
        
        if i == num_players - 1:
            position_from_button = 0  # Button
        else:
            position_from_button = i + 1  # SB=1, BB=2, UTG=3, etc.
        
        position_name = get_position_name(position_from_button, num_players)
        
        seat_rows.append(SeatRow(
            hand_id, seat_no, name, stack, ante,
            position_from_button, position_name
        ))
        seat_map[f'p{i+1}'] = name

    action_rows, showdown_rows, board_row = parse_actions_list(actions, seat_map)

    for row in action_rows:
        row.hand_id = hand_id
    for row in showdown_rows:
        row.hand_id = hand_id
    board_row.hand_id = hand_id

    dfs = {
        'hands': pd.DataFrame([asdict(hand_row)]),
        'seats': pd.DataFrame([asdict(r) for r in seat_rows]),
        'boards': pd.DataFrame([asdict(board_row)]),
        'actions': pd.DataFrame([asdict(r) for r in action_rows]),
        'showdown': pd.DataFrame([asdict(r) for r in showdown_rows]) if showdown_rows else pd.DataFrame(columns=TABLE_COLUMNS['showdown']),
    }
    return dfs

print("Normalization function loaded (with position calculation)")

In [ ]:
# File Processing: Parallel download with chunked Parquet saving
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

def empty_table(name: str) -> pd.DataFrame:
    return pd.DataFrame(columns=TABLE_COLUMNS[name])

def process_phhs_file(file_ref, hand_limit: Optional[int] = None) -> Dict[str, pd.DataFrame]:
    if isinstance(file_ref, dict):
        file_url = file_ref.get('download_url') or file_ref.get('url')
        source_label = file_ref.get('path', file_url)
    else:
        file_url = str(file_ref)
        source_label = file_url

    if not file_url:
        raise ValueError('Missing download URL for PHHS file')

    text = fetch_text(file_url)
    data = parse_phhs_to_dict(text)
    keys = sorted([k for k in data.keys() if k.isdigit()], key=lambda x: int(x))
    if hand_limit is not None:
        keys = keys[:hand_limit]

    per_table: Dict[str, List[pd.DataFrame]] = {name: [] for name in TABLE_ORDER}
    for key in keys:
        dfs = normalize_one_hand(data[key])
        for name in TABLE_ORDER:
            per_table[name].append(dfs[name])

    result = {}
    for name in TABLE_ORDER:
        if per_table[name]:
            df = pd.concat(per_table[name], ignore_index=True)
        else:
            df = empty_table(name)
        df = df.copy()
        df['source_file'] = source_label
        result[name] = df
    return result

def process_file_safe(file_ref, hand_limit=None):
    try:
        return process_phhs_file(file_ref, hand_limit)
    except Exception as exc:
        print(f"   Failed: {file_ref.get('path', file_ref)}: {exc}")
        return None

def collect_from_files_chunked(
    file_refs: List, 
    output_dir: Path,
    out_prefix: str,
    hand_limit: Optional[int] = None, 
    max_workers: int = 32,
    batch_size: int = 50
) -> Dict[str, int]:
    total = len(file_refs)
    row_counts = {name: 0 for name in TABLE_ORDER}
    
    # Create chunks directory
    chunks_dir = output_dir / 'chunks'
    chunks_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"   Parallel download: {max_workers} workers, saving chunks every {batch_size} files")
    print(f"   Chunks directory: {chunks_dir}")
    
    batch_num = 0
    for batch_start in range(0, total, batch_size):
        batch_end = min(batch_start + batch_size, total)
        batch_refs = file_refs[batch_start:batch_end]
        batch_results: Dict[str, List[pd.DataFrame]] = {name: [] for name in TABLE_ORDER}
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(process_file_safe, ref, hand_limit): ref for ref in batch_refs}
            for future in as_completed(futures):
                result = future.result()
                if result is not None:
                    for name in TABLE_ORDER:
                        batch_results[name].append(result[name])
        
        # Save each batch as separate Parquet file (avoids file size limits)
        for name in TABLE_ORDER:
            if batch_results[name]:
                batch_df = pd.concat(batch_results[name], ignore_index=True)
                chunk_path = chunks_dir / f"{out_prefix}_{name}_chunk_{batch_num:04d}.parquet"
                batch_df.to_parquet(chunk_path, index=False)
                row_counts[name] += len(batch_df)
        
        batch_num += 1
        print(f"   Progress: {batch_end}/{total} ({100*batch_end/total:.1f}%) - Chunk {batch_num} saved")
    
    return row_counts

def combine_chunks(output_dir: Path, out_prefix: str) -> Dict[str, int]:
    chunks_dir = output_dir / 'chunks'
    final_counts = {}
    
    for name in TABLE_ORDER:
        chunk_files = sorted(chunks_dir.glob(f"{out_prefix}_{name}_chunk_*.parquet"))
        if chunk_files:
            dfs = [pd.read_parquet(f) for f in chunk_files]
            combined = pd.concat(dfs, ignore_index=True)
            combined.to_parquet(output_dir / f"{out_prefix}_{name}.parquet", index=False)
            combined.to_csv(output_dir / f"{out_prefix}_{name}.csv", index=False)
            final_counts[name] = len(combined)
            print(f"   {name}: {len(combined):,} rows from {len(chunk_files)} chunks")
        else:
            final_counts[name] = 0
    return final_counts

print("Chunked Parquet processing loaded")

In [ ]:
# Execute ETL Pipeline
print("[1/3] Discovering PHHS files...")

try:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
except Exception as e:
    print(f"Warning: {e}")
    OUTPUT_DIR = Path('.')

selected_files = discover_file_urls(NUM_PARENT_FOLDERS, NUM_FILES_PER_FOLDER)
if not selected_files:
    raise ValueError('No PHHS files discovered')

print(f"[2/3] Processing {len(selected_files)} files (chunked Parquet)...")
row_counts = collect_from_files_chunked(
    selected_files, OUTPUT_DIR, OUT_PREFIX,
    HAND_LIMIT_PER_FILE, max_workers=32, batch_size=50
)

print("[3/3] Summary:")
for name in TABLE_ORDER:
    print(f"   {name}: {row_counts.get(name, 0):,} rows")

elapsed = time.time() - start_time
print(f"Runtime: {elapsed/60:.1f} minutes")
print(f"Chunks saved to: {OUTPUT_DIR / 'chunks'}")
print("[SUCCESS] Data ready - downstream notebooks will read chunks directly!")

In [ ]:
# Preview the collected data: Load and display first 10 rows of each table
print("\n" + "=" * 80)
print("DATA PREVIEW")
print("=" * 80)

for name in TABLE_ORDER:
    file_path = OUTPUT_DIR / f"{OUT_PREFIX}_{name}.csv"
    df = pd.read_csv(file_path)
    print(f"\n{name.upper()} ({len(df):,} total rows)")
    print("-" * 80)
    display(df.head(10))
    print(f"\nColumns: {', '.join(df.columns)}")
    print(f"Shape: {df.shape}")